# ECE1508: Deep Generative Models -- Summer 2026
## Assignment 3: Generative Adversarial Networks
## Question 3: GAN vs WGAN on MNIST

In this assignment, we train a vanilla GAN and WGAN on a small subset of MNIST containing only three digits. The goal is to compare numerical stability of vanilla GAN against WGAN.


### Loading Modules
Let's load some required modules.

In [ ]:
import random
from collections import Counter

import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset, TensorDataset
from torchvision import transforms
from torchvision.datasets import MNIST
from torchvision.utils import make_grid


# Device
if torch.backends.mps.is_available():
    device = torch.device('mps')
elif torch.cuda.is_available():
    device = torch.device('cuda')
else:
    device = torch.device('cpu')

print('Device:', device)



### Loading Dataset
We use digits `0`, `1`, and `2` to keep the experiment simple.


In [ ]:
# Sample 1000 samples of each digit
def load_three_digit_mnist(root='./data', digits=(0, 1, 2), max_per_digit=1000, train=True):
    transform = transforms.ToTensor()
    dataset = MNIST(root=root, train=train, download=True, transform=transform)
    targets = dataset.targets
    mask = torch.zeros_like(targets, dtype=torch.bool)
    ## COMPLETE ##



train_dataset = load_three_digit_mnist(train=True, max_per_digit=1000)


# Count labels in the subset
train_labels = []
for item in train_dataset:
    train_labels.append(int(item[1]))
print('Train label counts:', Counter(train_labels))


#### Data Loaders

Let's take a look at a few samples. 


In [ ]:
# batch size 
batch_size = 128

# build the loader
train_loader = ## COMPLETE ##

# Visualize a few real samples
fig, axes = plt.subplots(2, 8, figsize=(12, 4))
for ax, (img, label) in zip(axes.flatten(), list(train_dataset)[:16]):
    ## COMPLETE ##
plt.suptitle('Real MNIST samples -- Digits 0, 1, 2')
plt.tight_layout()
plt.show()


### Models (vanilla GAN)
We next implement the generator and discriminator. Clearly, the data in this case is of size $28 \times 28$. We consider a latent space of size $m=64$. 

#### Discriminator
The discriminator is to get a $28 \times 28$ input $x$ and classify it as `real` or `fake`. We consider the following MLP:
$$
\mathrm{Linear} (784 \rightarrow 128) + \mathrm{LeakyReLU}\\
\downarrow\\
\mathrm{Linear} (128 \rightarrow 128) + \mathrm{LeakyReLU}\\
\downarrow\\
\mathrm{Linear} (128 \rightarrow 128) + \mathrm{LeakyReLU}\\
\downarrow\\
\mathrm{Linear} (128 \rightarrow 1)
$$
For all $\mathrm{LeakyReLU}$ activations set $\texttt{negative\_slope}=0.2$. Note that the last layer should be properly activated for classification.


In [ ]:
class Discriminator(nn.Module):
    def __init__(self):
        ## COMPLETE ##

    def forward(self, x):
        ## COMPLETE ##


#### Generator

For generator, we need a network which converts a 64-dimensional latent to a $28 \times 28$ MNIST image. For this, we implement the following MLP:
$$
\mathrm{Linear} (64 \rightarrow 128) + \mathrm{ReLU}\\
\downarrow\\
\mathrm{Linear} (128 \rightarrow 128) + \mathrm{ReLU}\\
\downarrow\\
\mathrm{Linear} (128 \rightarrow 128) + \mathrm{ReLU}\\
\downarrow\\
\mathrm{Linear} (128 \rightarrow 784) + \mathrm{Sigmoid}
$$


In [ ]:

class Generator(nn.Module):
    def __init__(self):
        ## COMPLETE ##

    def forward(self, z):
        ## COMPLETE ##


#### Sampling the Generator

We finally write a function to sample the generator.


In [ ]:
# Sample latent
def sample_noise(n, device):
    ## COMPLETE ##


# Pass it through the generator
def show_generated_images(generator, title='Generated samples', n=16):
    ## COMPLETE ##



Let's initiate a random generator and look at its samples before training.

In [ ]:
G = Generator().to(device)
show_generated_images(G, title='Generated samples', n=16)



### Training via Vanilla GAN Loop

We now train this model as we did for Swiss Roll dataset, i.e., with the vanilla GAN. 


In [ ]:
def train_GAN_vanilla(n_inner = 6, epochs = 100):
    # define right loss
    ## COMPLETE ##
    
    # instantiate G and D
    G = Generator().to(device)
    D = Discriminator().to(device)

    # Set the optimizers
    lr_D = 2e-3
    lr_G = 2e-4
    opt_G = torch.optim.Adam(G.parameters(), lr=lr_G, betas=(0.5, 0.999))
    opt_D = torch.optim.Adam(D.parameters(), lr=lr_D, betas=(0.5, 0.999))

    history = {
        'd_loss': [],
        'g_loss': [],
        'd_real': [],
        'd_fake': [],
    }

    # Start the loop
    for epoch in range(1, epochs + 1):
        G.train()
        D.train()

        d_loss_epoch = 0.0
        g_loss_epoch = 0.0
        d_real_epoch = 0.0
        d_fake_epoch = 0.0
        num_batches = 0

        for x_real, _ in train_loader:
            ## COMPLETE ##

            # inner loop for discriminator
            for _ in range(n_inner):
                ## COMPLETE ##

            # Update the generator
            ## COMPLETE ##

            with torch.no_grad():
                d_real_prob = ## COMPLETE ##
                d_fake_prob = ## COMPLETE ##

            d_loss_epoch += ## COMPLETE ##
            g_loss_epoch += ## COMPLETE ##
            d_real_epoch += ## COMPLETE ##
            d_fake_epoch += ## COMPLETE ##
            num_batches += ## COMPLETE ##

        # Update history
        ## COMPLETE ##

        if epoch == 1 or epoch % 20 == 0:
            print(f'Epoch {epoch:03d}/{epochs} | D loss: {history["d_loss"][-1]:.4f} | G loss: {history["g_loss"][-1]:.4f} | D(real): {history["d_real"][-1]:.3f} | D(fake): {history["d_fake"][-1]:.3f}')
            
    return G, history


#### Experiments with Vanilla GAN

Let's now train the model and look at its output samples.


In [ ]:
# Train
G, history = train_GAN_vanilla()

# Sample the trained model
show_generated_images(G, title='Generated Samples by Vanilla GAN')

## Question: _What do you observation?Do a quick research to understand what this behavior is called._
_## COMPLETE ##_

#### Training Dynamics of Vanilla GAN
Let's look at the losses

In [ ]:
plt.figure(figsize=(8, 4))
## COMPLETE ##
plt.legend()
plt.show()

## Question: _Does the loss curve tell you anything in support of your earlier observation?_
_## COMPLETE ##_

### Confidence of Vanilla GAN
Let's look at the confidence at discriminator.

In [ ]:
plt.figure(figsize=(8, 4))
## COMPLETE ##
plt.legend()
plt.show()

## Question: _What does the confidence curve say in support of your earlier observation?_
_## COMPLETE ##_

### Training via WGAN Loop
We now modify the training loop for WGAN apprach. Note that we do not need to change the generator or discriminator. It's enough to drop the final activation of the discriminator. 

__Lipschitz Continuity:__ To guarantee Lipschitz continuity of the discriminator, we use weight clipping. To this end, after each update of discriminator parameters, we clip them as
$$ \phi \gets \mathrm{Clip}(\phi, 0.01) = 
\begin{cases}
\phi_i &\vert \phi_i \vert \leq 0.01\\
\mathrm{sign}(\phi_i) 0.01 &\vert \phi_i \vert > 0.01
\end{cases}
$$
To apply this clipping, we could directly use the method `.data.clamp_(-clip_value, clip_value)` with `clip_value = 0.01`.


#### Modifying Train Loop
Modify the train loop to implement WGAN training. 

In [ ]:
def train_WGAN(n_inner = 6, epochs = 100):
    # instantiate G and D
    ## COMPLETE ## You may copy your previous training loop and modify it directly here

        if epoch == 1 or epoch % 20 == 0:
            print(f'Epoch {epoch:03d}/{epochs} | D loss: {history["d_loss"][-1]:.4f} | G loss: {history["g_loss"][-1]:.4f} | D(real): {history["d_real"][-1]:.3f} | D(fake): {history["d_fake"][-1]:.3f}')
            
    return G, history

#### Experiment with WGAN
Let's now look at samples given by WGAN after training.

In [ ]:
# Train
G, history = train_WGAN()

# Sample the trained model
show_generated_images(G, title='Generated Samples by WGAN')

## Question: _Any change in the previously seen behavior? Explain your observation. What is your conclusion?_
_## COMPLETE ##_

#### Training Dynamics of WGAN
Now look at the training dynamics.

In [ ]:
plt.figure(figsize=(8, 4))
## COMPLETE ##
plt.legend()
plt.show()

## Question: _Does this observation aligns with your conclusion? Explain._
_## COMPLETE ##_